# LeetCode 3875 — Construct Uniform Parity Array I bằng Python

Link đề: https://leetcode.com/problems/construct-uniform-parity-array-i/ · Độ khó: **Easy** · Chủ đề: Math, Parity, Constructive, Brainteaser

Skeleton LeetCode:

```python
class Solution:
    def uniformArray(self, nums1: List[int]) -> bool:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints, chữ ký hàm và lời giải "luôn trả về `True`" đã được đối chiếu qua leetcode.doocs.org (Weekly Contest 494, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình nên cần đối chiếu từng chữ với đề gốc nếu muốn chắc chắn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ mảng `nums1` gồm các số **phân biệt**, dựng mảng `nums2` mà mỗi phần tử hoặc là `nums1[i]`, hoặc là `nums1[i] - nums1[j]` (với `j != i`), sao cho **mọi phần tử của `nums2` cùng tính chẵn/lẻ**.
2. Nắm các **quy tắc tính chẵn/lẻ** của phép trừ: `chẵn - chẵn = chẵn`, `lẻ - lẻ = chẵn`, `chẵn - lẻ = lẻ`, `lẻ - chẵn = lẻ`.
3. Biết cách **chứng minh bằng xây dựng (constructive)**: chỉ ra một cách dựng `nums2` luôn thành công, từ đó kết luận đáp án luôn là `True`.
4. Viết 5 cách khác nhau: vét cạn mọi lựa chọn (với `n` nhỏ), giao các tập parity có thể đạt, kiểm tra theo parity đích bằng đếm, dựng mảng rồi kiểm chứng, và lời giải `O(1)`.
5. Tránh lỗi: nghĩ rằng chỉ đúng khi mảng sẵn cùng parity, nhầm `chẵn - lẻ` là chẵn, dùng giá trị **đã đổi** thay vì giá trị gốc của `nums1`.

## 2. Đề bài & ví dụ

Cho mảng `nums1` gồm `n` số nguyên **đôi một khác nhau**. Bạn cần dựng mảng `nums2` có cùng độ dài, trong đó với mỗi chỉ số `i`, `nums2[i]` phải là **một trong hai** giá trị:

- `nums1[i]` (giữ nguyên), hoặc
- `nums1[i] - nums1[j]` với một chỉ số `j` **khác** `i` nào đó.

Trả về `true` nếu có thể dựng `nums2` sao cho **mọi phần tử đều lẻ** hoặc **mọi phần tử đều chẵn**; ngược lại trả `false`.

| Ví dụ | nums1 | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,3]` | `true` | Chọn `nums2[0] = 2 - 3 = -1`, `nums2[1] = 3`; mảng `[-1, 3]` toàn số lẻ |
| 2 | `[4,6]` | `true` | Giữ nguyên `[4, 6]`, toàn số chẵn |

Constraints: `1 <= n == nums1.length <= 100`; `1 <= nums1[i] <= 100`; các phần tử của `nums1` đôi một khác nhau.

In [1]:
nums1 = [2, 3]
print("2 - 3 =", 2 - 3, " ->", (2 - 3) % 2, "(lẻ nếu = 1)")
print("[2,3] parity:", [x % 2 for x in nums1])
print("[-1, 3] parity:", [x % 2 for x in [-1, 3]])

2 - 3 = -1  -> 1 (lẻ nếu = 1)
[2,3] parity: [0, 1]
[-1, 3] parity: [1, 1]


## 3. Kiến thức nền

### 3.1. Chẵn/lẻ (parity) và toán tử `%`

`x % 2` là `0` nếu `x` chẵn và `1` nếu `x` lẻ. Với số **âm** Python vẫn cho kết quả `0` hoặc `1` (`-1 % 2 == 1`), nên kiểm tra lẻ bằng `x % 2 == 1` là an toàn.

In [2]:
print([x % 2 for x in [4, 7, -1, -4, 0]])

[0, 1, 1, 0, 0]


### 3.2. Bảng parity của phép trừ

Chỉ cần xét parity: `a - b` có parity của `(a + b) % 2` (vì trừ và cộng cùng parity). Bảng (0 = chẵn, 1 = lẻ):

| a | b | a - b |
|---|---|---|
| 0 | 0 | 0 (chẵn) |
| 1 | 1 | 0 (chẵn) |
| 0 | 1 | 1 (lẻ) |
| 1 | 0 | 1 (lẻ) |

Đó là phép **XOR** của hai bit parity.

In [3]:
for a in (0, 1):
    for b in (0, 1):
        print(f"parity {a} - parity {b} -> {(a - b) % 2} = XOR {a ^ b}")
print((7 - 4) % 2, (4 - 7) % 2, (7 - 3) % 2, (4 - 6) % 2)

parity 0 - parity 0 -> 0 = XOR 0
parity 0 - parity 1 -> 1 = XOR 1
parity 1 - parity 0 -> 1 = XOR 1
parity 1 - parity 1 -> 0 = XOR 0
1 1 0 0


### 3.3. `itertools.product`

`product(*danh_sách)` sinh tích Descartes: mọi cách chọn một phần tử từ mỗi danh sách. Ta dùng nó để vét cạn mọi cách chọn `nums2` khi `n` nhỏ.

In [4]:
from itertools import product
print(list(product([1, 2], [10, 20])))
print(len(list(product(range(3), repeat=3))))

[(1, 10), (1, 20), (2, 10), (2, 20)]
27


### 3.4. `all`, `any` và tập hợp (`set`)

`all(iterable)` đúng khi mọi phần tử đúng; `any` đúng khi có ít nhất một. `set` giúp kiểm tra "mọi phần tử cùng một giá trị": `len(set(parities)) == 1`. Giao hai tập: `A & B`.

In [5]:
par = [1, 1, 1]
print(all(p == 1 for p in par), any(p == 0 for p in par), len(set(par)) == 1)
print({0, 1} & {1}, {0} & {1}, set().union({0}, {1}))

True False True
{1} set() {0, 1}


### 3.5. Đếm số phần tử lẻ

`sum(x % 2 for x in nums)` cho số phần tử lẻ (vì mỗi số lẻ góp `1`). Số phần tử chẵn là `len(nums)` trừ số này.

In [6]:
nums = [2, 3, 8, 5]
odd = sum(x % 2 for x in nums)
print("số lẻ:", odd, "số chẵn:", len(nums) - odd)

số lẻ: 2 số chẵn: 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chỉ **parity** mới quan trọng, nên ta nghĩ trong thế giới hai trạng thái `chẵn` (E) và `lẻ` (O). Với mỗi phần tử `nums1[i]`, mục tiêu là đưa nó về cùng một parity đích. Có hai nhóm phần tử:

- phần tử **đã có parity đích**: giữ nguyên;
- phần tử **sai parity**: phải dùng `nums1[i] - nums1[j]` để đổi parity. Phép trừ **đổi parity** khi `nums1[j]` lẻ (vì `a - lẻ` luôn có parity ngược `a`), và **giữ parity** khi `nums1[j]` chẵn.

Vậy: để đưa một số **chẵn** thành **lẻ** ta trừ đi một số **lẻ** bất kỳ (khác chính nó, và vì nó chẵn nên mọi số lẻ đều khác nó). Để đưa phần tử **lẻ** thành **chẵn** ta trừ đi một số **lẻ khác**.

Thử `[2, 3, 8]`: chọn đích = lẻ. `3` đã lẻ, giữ. `2` chẵn: `2 - 3 = -1` (lẻ). `8` chẵn: `8 - 3 = 5` (lẻ). Kết quả `[-1, 3, 5]` toàn lẻ. Còn nếu mọi phần tử đều chẵn (`[2, 4, 6]`) thì giữ nguyên, đã thỏa.

Gần như luôn có cách. Câu hỏi: có trường hợp nào **thất bại** không? Ta sẽ chứng minh là **không** (mục 6).

In [7]:
nums1 = [2, 3, 8]
odd_value = next(x for x in nums1 if x % 2 == 1)           # một số lẻ làm "chất đổi parity"
nums2 = [x if x % 2 == 1 else x - odd_value for x in nums1]
print(nums2, [x % 2 for x in nums2])

[-1, 3, 5] [1, 1, 1]


## 5. Cách 1 — Brute Force / baseline

Vét cạn: với mỗi chỉ số `i`, tập lựa chọn là `[nums1[i]] + [nums1[i] - nums1[j] for j != i]` (tối đa `n` lựa chọn). Dùng `product` để thử **mọi** `nums2` có thể, và kiểm tra có mảng nào toàn cùng parity. Chỉ chạy được khi `n` rất nhỏ (có `n^n` khả năng), nhưng là **chân lý** để kiểm chứng.

Nút thắt: `O(n^n)` hoàn toàn không thể dùng với `n = 100`; ta sẽ thay bằng lập luận parity.

In [8]:
from itertools import product

class SolutionBrute:
    def uniformArray(self, nums1):
        n = len(nums1)
        options = []
        for i in range(n):
            opts = [nums1[i]] + [nums1[i] - nums1[j] for j in range(n) if j != i]
            options.append(opts)
        for nums2 in product(*options):
            if len({x % 2 for x in nums2}) == 1:
                return True
        return False

print(SolutionBrute().uniformArray([2, 3]), SolutionBrute().uniformArray([4, 6]), SolutionBrute().uniformArray([5]))

True True True


**Độ phức tạp:** Time `O(n^n · n)`, chỉ dùng cho `n <= 6`. Space `O(n²)` cho các lựa chọn.

## 6. Ý tưởng thuật toán / bước đột phá

**Chứng minh luôn đúng bằng xây dựng:**

- **Trường hợp A:** mọi phần tử đã cùng parity. Giữ nguyên `nums2 = nums1` -> thỏa.
- **Trường hợp B:** có cả chẵn và lẻ. Chọn đích **lẻ**. Gọi `o` là một phần tử lẻ của `nums1`.
  - Phần tử lẻ: giữ nguyên (đã lẻ).
  - Phần tử chẵn `e`: đặt `nums2[i] = e - o`. Đây là **chẵn - lẻ = lẻ**, và `o` ở chỉ số khác `i` (vì `e` chẵn, `o` lẻ) nên hợp lệ.

Cả hai trường hợp đều dựng được, nên **đáp án luôn là `True`** với mọi `n >= 1` (kể cả `n = 1`, mảng một phần tử tự nó đã đồng nhất).

**Invariant** của việc dựng: mọi `nums2[i]` thuộc hai lựa chọn hợp lệ của đề và có cùng parity đích.

Các cách thực hiện thuật toán: (2) giao tập parity có thể đạt của từng chỉ số — không cần biết đáp án luôn đúng; (3) kiểm tra theo parity đích bằng đếm; (4) dựng mảng rồi kiểm chứng; (5) trả thẳng `True`.

## 7. Các cách tiếp theo

### Cách 2 — Giao các tập parity có thể đạt

Với mỗi `i`, tập parity có thể đạt của `nums2[i]` là `R_i = {parity(nums1[i])} ∪ {parity(nums1[i] - nums1[j]) : j != i}`. Có đáp án khi giao `R_0 ∩ R_1 ∩ ... ∩ R_{n-1}` khác rỗng (ta chọn parity đích trong giao). Không cần biết trước đáp án luôn `True`.

In [9]:
class SolutionReach:
    def uniformArray(self, nums1):
        n = len(nums1)
        common = {0, 1}
        for i in range(n):
            reach = {nums1[i] % 2}
            for j in range(n):
                if j != i:
                    reach.add((nums1[i] - nums1[j]) % 2)
            common &= reach
        return len(common) > 0

print(SolutionReach().uniformArray([2, 3]), SolutionReach().uniformArray([4, 6]), SolutionReach().uniformArray([5]))

True True True


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

### Cách 3 — Kiểm tra từng parity đích bằng đếm

Gọi `odd` là số phần tử lẻ, `even = n - odd`.

- **Đích lẻ** làm được khi mọi phần tử chẵn đều có một số lẻ để trừ: `even == 0` hoặc `odd >= 1`.
- **Đích chẵn** làm được khi mọi phần tử lẻ có **một số lẻ khác** để trừ: `odd == 0` hoặc `odd >= 2`.

Đáp án là `can_odd or can_even`.

In [10]:
class SolutionCount:
    def uniformArray(self, nums1):
        odd = sum(x % 2 for x in nums1)
        even = len(nums1) - odd
        can_odd = even == 0 or odd >= 1
        can_even = odd == 0 or odd >= 2
        return can_odd or can_even

print(SolutionCount().uniformArray([2, 3]), SolutionCount().uniformArray([4, 6]), SolutionCount().uniformArray([5]))

True True True


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Dựng `nums2` thật rồi kiểm chứng

Làm đúng theo chứng minh ở mục 6: nếu có số lẻ `o` thì mọi số chẵn `e` thành `e - o`; sau đó **kiểm tra** `nums2` thỏa mọi điều kiện của đề (mỗi phần tử là `nums1[i]` hoặc `nums1[i] - nums1[j]` với `j != i`, và cùng parity). Hàm trả về kết quả kiểm chứng.

In [11]:
class SolutionConstruct:
    def build(self, nums1):
        # trả về (nums2, chỉ số j đã dùng cho từng i) theo chứng minh ở mục 6
        k = next((i for i, x in enumerate(nums1) if x % 2 == 1), None)
        if k is None:
            return list(nums1), [None] * len(nums1)
        nums2, used = [], []
        for i, x in enumerate(nums1):
            if x % 2 == 1:
                nums2.append(x); used.append(None)
            else:
                nums2.append(x - nums1[k]); used.append(k)
        return nums2, used

    def uniformArray(self, nums1):
        nums2, used = self.build(nums1)
        for i, (v, j) in enumerate(zip(nums2, used)):
            if j is None:
                assert v == nums1[i]
            else:
                assert j != i and v == nums1[i] - nums1[j]
        return len({v % 2 for v in nums2}) == 1

print(SolutionConstruct().uniformArray([2, 3]), SolutionConstruct().uniformArray([4, 6]), SolutionConstruct().uniformArray([5]))
print(SolutionConstruct().build([2, 3, 8]))

True True True
([-1, 3, 5], [1, None, 1])


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 5 — Lời giải `O(1)`: luôn `True`

Vì lập luận ở mục 6 chứng minh luôn tồn tại cách dựng, hàm chỉ cần `return True`.

In [12]:
class SolutionAlwaysTrue:
    def uniformArray(self, nums1):
        return True

print(SolutionAlwaysTrue().uniformArray([2, 3]), SolutionAlwaysTrue().uniformArray([4, 6]))

True True


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 và Cách 4 trên `nums1 = [2, 3, 8]`: `odd = 1`, `even = 2`.

| Đích | điều kiện | giá trị | làm được? |
|---|---|---|---|
| lẻ | `even == 0 or odd >= 1` | `False or True` | có |
| chẵn | `odd == 0 or odd >= 2` | `False or False` | không |

Đích lẻ làm được -> trả `True`. Dựng: số lẻ `o = 3` (chỉ số 1); `2 -> 2 - 3 = -1`; `3` giữ; `8 -> 8 - 3 = 5`. `nums2 = [-1, 3, 5]`.

In [13]:
def trace_count(nums1):
    odd = sum(x % 2 for x in nums1)
    even = len(nums1) - odd
    can_odd = even == 0 or odd >= 1
    can_even = odd == 0 or odd >= 2
    print(f"nums1={nums1}: lẻ={odd} chẵn={even}")
    print(f"  đích lẻ: even==0? {even == 0}, odd>=1? {odd >= 1} -> {can_odd}")
    print(f"  đích chẵn: odd==0? {odd == 0}, odd>=2? {odd >= 2} -> {can_even}")
    return can_odd or can_even

assert trace_count([2, 3, 8]) is True
assert trace_count([2, 4]) is True
assert trace_count([1, 2]) is True
assert trace_count([7]) is True

nums1=[2, 3, 8]: lẻ=1 chẵn=2
  đích lẻ: even==0? False, odd>=1? True -> True
  đích chẵn: odd==0? False, odd>=2? False -> False
nums1=[2, 4]: lẻ=0 chẵn=2
  đích lẻ: even==0? False, odd>=1? False -> False
  đích chẵn: odd==0? True, odd>=2? False -> True
nums1=[1, 2]: lẻ=1 chẵn=1
  đích lẻ: even==0? False, odd>=1? True -> True
  đích chẵn: odd==0? False, odd>=2? False -> False
nums1=[7]: lẻ=1 chẵn=0
  đích lẻ: even==0? True, odd>=1? True -> True
  đích chẵn: odd==0? False, odd>=2? False -> False


In [14]:
def trace_build(nums1):
    k = next((i for i, x in enumerate(nums1) if x % 2 == 1), None)
    print(f"nums1={nums1}, chỉ số số lẻ dùng làm chất đổi parity: {k}")
    nums2 = []
    for i, x in enumerate(nums1):
        if k is None or x % 2 == 1:
            nums2.append(x)
            print(f"  i={i}: giữ {x} (parity {x % 2})")
        else:
            nums2.append(x - nums1[k])
            print(f"  i={i}: {x} - {nums1[k]} = {nums2[-1]} (parity {nums2[-1] % 2})")
    print("  nums2 =", nums2, "-> parity", [v % 2 for v in nums2])
    return nums2

res = trace_build([2, 3, 8])
assert len({v % 2 for v in res}) == 1
res = trace_build([4, 6, 10])
assert len({v % 2 for v in res}) == 1

nums1=[2, 3, 8], chỉ số số lẻ dùng làm chất đổi parity: 1
  i=0: 2 - 3 = -1 (parity 1)
  i=1: giữ 3 (parity 1)
  i=2: 8 - 3 = 5 (parity 1)
  nums2 = [-1, 3, 5] -> parity [1, 1, 1]
nums1=[4, 6, 10], chỉ số số lẻ dùng làm chất đổi parity: None
  i=0: giữ 4 (parity 0)
  i=1: giữ 6 (parity 0)
  i=2: giữ 10 (parity 0)
  nums2 = [4, 6, 10] -> parity [0, 0, 0]


## 9. Code nộp LeetCode

Dùng lời giải ngắn nhất, đã được chứng minh ở mục 6 (và đối chiếu với các cách khác ở mục 12).

In [15]:
from typing import List, Optional


class Solution:
    def uniformArray(self, nums1: List[int]) -> bool:
        # Luôn dựng được: nếu có số lẻ o thì đưa mọi số chẵn e thành e - o (lẻ);
        # nếu không có số lẻ thì nums1 đã toàn chẵn.
        return True


print(Solution().uniformArray([2, 3]), Solution().uniformArray([4, 6]))

True True


## 10. Vì sao thuật toán đúng

1. **Chỉ có hai trường hợp.** Hoặc `nums1` không có số lẻ (toàn chẵn), hoặc có ít nhất một số lẻ.
2. **Không có số lẻ.** `nums2 = nums1` thỏa: mọi phần tử đều chẵn.
3. **Có số lẻ `o`.** Mọi số lẻ giữ nguyên (lẻ). Mọi số chẵn `e` thay bằng `e - o`: do `e` chẵn và `o` lẻ nên `e - o` lẻ. Chỉ số của `e` và `o` khác nhau vì một số chẵn một số lẻ, nên lựa chọn hợp lệ theo đề. `nums2` toàn lẻ.
4. **Trường hợp `n = 1`.** Chỉ có `nums1[0]`, giữ nguyên, mảng một phần tử luôn đồng nhất parity.
5. Các lời giải tổng quát hơn (Cách 1, 2, 3, 4) cho cùng kết quả `True`, điều này được kiểm chứng ở mục 12. Phép trừ chỉ dùng giá trị **gốc** `nums1[j]` (không phải `nums2[j]`), đúng như đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums1)`.

- Cách 1: `O(n^n · n)` thời gian, chỉ cho `n <= 6`; `O(n²)` bộ nhớ.
- Cách 2: `O(n²)` thời gian, `O(1)` bộ nhớ.
- Cách 3: `O(n)` thời gian, `O(1)` bộ nhớ.
- Cách 4: `O(n)` thời gian, `O(n)` bộ nhớ.
- Cách 5 (nộp bài): `O(1)` thời gian và bộ nhớ.

Khác với đa số bài, ở đây độ phức tạp thấp nhất đạt được nhờ **chứng minh toán học** chứ không phải cấu trúc dữ liệu. Vì đáp án không phụ thuộc dữ liệu nên best/average/worst đều như nhau.

## 12. Bộ test

Đối chiếu 5 cách với ví dụ đề, biên (`n = 1`, toàn chẵn, toàn lẻ, hỗn hợp) và nhiều input ngẫu nhiên (các phần tử đôi một khác nhau, `1..100`). Cách 1 (vét cạn) chỉ chạy với `n <= 5`.

In [16]:
import random

fast = [SolutionReach(), SolutionCount(), SolutionConstruct(), SolutionAlwaysTrue(), Solution()]

fixed = [[2, 3], [4, 6], [5], [2], [1, 3], [1, 3, 5], [2, 4, 6], [1, 2], [100, 99, 98], [7, 8, 9, 10], [1, 100]]
for nums1 in fixed:
    for s in fast + [SolutionBrute()]:
        assert s.uniformArray(list(nums1)) is True, (type(s).__name__, nums1)

random.seed(38750)
for _ in range(300):
    n = random.randint(1, 100)
    nums1 = random.sample(range(1, 101), n)
    outs = [s.uniformArray(list(nums1)) for s in fast]
    assert all(o is True for o in outs), nums1

for _ in range(150):
    n = random.randint(1, 5)
    nums1 = random.sample(range(1, 101), n)
    assert SolutionBrute().uniformArray(list(nums1)) is True
    assert SolutionReach().uniformArray(list(nums1)) is True
print("tất cả test OK")

tất cả test OK


### Test thuộc tính (kiểm chứng phép dựng)

Với mỗi mảng ngẫu nhiên, hàm `build` ở Cách 4 phải cho `nums2` thỏa đề: cùng parity, và mỗi `nums2[i]` bằng `nums1[i]` hoặc `nums1[i] - nums1[j]` với `j != i`.

In [17]:
random.seed(3875)
builder = SolutionConstruct()
for _ in range(300):
    n = random.randint(1, 100)
    nums1 = random.sample(range(1, 101), n)
    nums2, used = builder.build(nums1)
    assert len(nums2) == n and len({v % 2 for v in nums2}) == 1
    for i in range(n):
        ok = nums2[i] == nums1[i] or any(nums2[i] == nums1[i] - nums1[j] for j in range(n) if j != i)
        assert ok
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: kết luận "chỉ đúng khi mảng sẵn cùng parity"

Nhiều bạn quên rằng có thể **trừ** để đổi parity. Với `[2, 3]` kiểm tra "đã đồng nhất chưa?" trả `False`, nhưng đáp án đúng là `True`.

In [18]:
def wrong_already_uniform(nums1):
    return len({x % 2 for x in nums1}) == 1                    # SAI: bỏ qua phép trừ

print(wrong_already_uniform([2, 3]), "!=", Solution().uniformArray([2, 3]))
assert wrong_already_uniform([2, 3]) is False and Solution().uniformArray([2, 3]) is True
assert wrong_already_uniform([4, 6]) is True

False != True


### Lỗi 2: nhầm parity của `chẵn - lẻ`

`chẵn - lẻ` là **lẻ**, không phải chẵn. Nếu tính sai, ta tưởng không thể đổi parity bằng phép trừ.

In [19]:
def wrong_parity_rule(nums1):
    # giả sử (sai) rằng phép trừ luôn giữ nguyên parity của số bị trừ
    return len({x % 2 for x in nums1}) == 1

print((2 - 3) % 2, (3 - 2) % 2, (2 - 4) % 2)
assert (2 - 3) % 2 == 1 and (3 - 2) % 2 == 1 and (2 - 4) % 2 == 0
assert wrong_parity_rule([1, 2]) is False and Solution().uniformArray([1, 2]) is True

1 1 0


### Lỗi 3: chỉ thử đích **chẵn** rồi kết luận `False`

Có hai parity đích. Nếu chỉ thử "toàn chẵn" (cần `odd == 0` hoặc `odd >= 2`) thì với một số lẻ duy nhất ta kết luận sai, quên mất đích **lẻ** vẫn làm được.

In [20]:
def wrong_only_even_target(nums1):
    odd = sum(x % 2 for x in nums1)
    return odd == 0 or odd >= 2                               # SAI: bỏ qua đích lẻ

print(wrong_only_even_target([1, 2]), "!=", Solution().uniformArray([1, 2]))
assert wrong_only_even_target([1, 2]) is False and Solution().uniformArray([1, 2]) is True
assert wrong_only_even_target([2, 3, 8]) is False and Solution().uniformArray([2, 3, 8]) is True

False != True


### Lỗi 4: quên điều kiện `j != i`

Nếu cho phép `j = i` thì `nums1[i] - nums1[i] = 0` (chẵn) luôn biến mọi phần tử thành chẵn kể cả khi `n = 1`. Đề cấm điều này; với `n >= 2` kết quả vẫn `True` nhưng lý do sai. Kiểm tra bằng brute force có/không có `j != i`: các lựa chọn khác nhau.

In [21]:
def options_wrong(nums1, i):
    return [nums1[i]] + [nums1[i] - nums1[j] for j in range(len(nums1))]          # SAI: cho phép j == i

def options_right(nums1, i):
    return [nums1[i]] + [nums1[i] - nums1[j] for j in range(len(nums1)) if j != i]

print(options_wrong([5], 0), options_right([5], 0))
assert options_wrong([5], 0) == [5, 0] and options_right([5], 0) == [5]
assert 0 in options_wrong([5, 7], 0) and 0 not in options_right([5, 7], 0)

[5, 0] [5]


### Lỗi 5: tự thêm điều kiện "số lượng lẻ phải chẵn"

Một số bạn đoán "đáp án là `True` khi số phần tử lẻ chẵn". Điều này sai với `[2, 3]` (một số lẻ nhưng đáp án vẫn `True`).

In [22]:
def wrong_odd_count_even(nums1):
    return sum(x % 2 for x in nums1) % 2 == 0                  # SAI: suy đoán không có cơ sở

print(wrong_odd_count_even([2, 3]), "!=", Solution().uniformArray([2, 3]))
assert wrong_odd_count_even([2, 3]) is False and Solution().uniformArray([2, 3]) is True
assert wrong_odd_count_even([1, 3, 5]) is False and Solution().uniformArray([1, 3, 5]) is True

False != True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "có thể dựng ... không?" trên số nhỏ (`n <= 100`) mà điều kiện quá rộng -> hãy nghi ngờ đáp án **luôn `true`** và thử chứng minh bằng xây dựng.
- Bài về chẵn/lẻ: chuyển hết sang bit parity, phép trừ/cộng = XOR.
- Khi không chắc, viết brute force `n` nhỏ (hoặc `product`) làm **chân lý** rồi quan sát quy luật.
- "Bất kỳ phần tử nào khác chính nó" thường nghĩa là chỉ cần **một** phần tử "đặc biệt" (ở đây là một số lẻ) để dùng cho mọi phần tử còn lại.
- Đừng để tên bài "Uniform Parity" đánh lừa: phần phức tạp thực sự ở phiên bản II (có thêm ràng buộc) chứ không phải bản I.

## 15. Biến thể của bài

1. **Construct Uniform Parity Array II** (bài cùng cặp): thêm ràng buộc khiến không còn luôn `true`.
2. **Sort Array By Parity** (905), **Sort Array By Parity II** (922): sắp xếp theo parity.
3. **Single Number** (136): XOR/parity để tìm phần tử lẻ lần.
4. **Minimum Operations to Make Array Parity Same**-style: số thao tác tối thiểu để mọi phần tử cùng parity.
5. **Check If Array Is Good** / các bài "luôn đúng" (brainteaser) ở các cuộc thi Weekly.
6. Biến thể: nếu chỉ được trừ **số bé hơn** hoặc chỉ dùng **số kề**: bài toán mới, cần xét lại.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `product` mọi `nums2` | `O(n^n · n)` | `O(n²)` | Chân lý cho `n <= 6` |
| 2. Giao tập parity | `R_i` rồi giao | `O(n²)` | `O(1)` | Không cần biết đáp án luôn đúng |
| 3. Đếm số lẻ | điều kiện đích lẻ/chẵn | `O(n)` | `O(1)` | Nhìn rõ cấu trúc |
| 4. Dựng + kiểm chứng | dùng một số lẻ làm "chất đổi" | `O(n)` | `O(n)` | Chứng minh bằng xây dựng |
| 5. `return True` | kết luận luôn đúng | `O(1)` | `O(1)` | Nộp bài |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().uniformArray([1])` và `Solution().uniformArray([10, 20, 30])` ra gì?

**Bài 2 (tính tay):** với `nums1 = [4, 5, 6]`, hãy dựng `nums2` toàn lẻ rồi một `nums2` toàn chẵn (nếu được).

**Bài 3 (điền TODO):** hoàn thành `make_all_odd(nums1)` trả về một `nums2` toàn lẻ (giả sử `nums1` có ít nhất một số lẻ).

**Bài 4 (tự viết lại):** viết `make_all_even(nums1)` trả về một `nums2` toàn chẵn nếu có thể, ngược lại `None` (gợi ý: cần `odd == 0` hoặc `odd >= 2`).

**Bài 5 (mini challenge):** viết `count_changes_to_all_odd(nums1)` trả về số phần tử phải đổi (thay bằng hiệu) để toàn lẻ, hoặc `-1` nếu không thể (khi `nums1` không có số lẻ và `n >= 1`).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().uniformArray([1]), Solution().uniformArray([10, 20, 30]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def make_all_odd_todo(nums1):
    nums2 = list(nums1)
    # TODO: tìm một số lẻ o, đổi mọi số chẵn e thành e - o
    return nums2

True True


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: một phần tử luôn đồng nhất; `[10, 20, 30]` toàn chẵn. Bài 2: `5` là số lẻ duy nhất.

**Mức 2 (vừa):** Bài 3: `o = next(x for x in nums1 if x % 2 == 1)`. Bài 4: nếu `odd >= 2` thì mọi số lẻ `x` thành `x - o` với `o` là một số lẻ **khác chỉ số**. Bài 5: số phần tử chẵn chính là số phép đổi.

**Mức 3 (gần đáp án):** Bài 2: toàn lẻ: `[4 - 5, 5, 6 - 5] = [-1, 5, 1]`; toàn chẵn: không được vì chỉ có một số lẻ `5` (không có số lẻ khác để trừ). Bài 5: `count_changes_to_all_odd([2, 3, 8]) = 2`; `[2, 4]` cho `-1`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().uniformArray([1]) is True and Solution().uniformArray([10, 20, 30]) is True

# Bài 3
def make_all_odd(nums1):
    o = next(x for x in nums1 if x % 2 == 1)
    return [x if x % 2 == 1 else x - o for x in nums1]
res = make_all_odd([4, 5, 6])
assert res == [-1, 5, 1] and all(v % 2 == 1 for v in res)

# Bài 4
def make_all_even(nums1):
    odd_idx = [i for i, x in enumerate(nums1) if x % 2 == 1]
    if not odd_idx:
        return list(nums1)
    if len(odd_idx) < 2:
        return None
    res = []
    for i, x in enumerate(nums1):
        if x % 2 == 0:
            res.append(x)
        else:
            j = odd_idx[0] if odd_idx[0] != i else odd_idx[1]
            res.append(x - nums1[j])
    return res
assert make_all_even([4, 5, 6]) is None
r = make_all_even([1, 3, 6])
assert r is not None and all(v % 2 == 0 for v in r)
assert make_all_even([2, 4]) == [2, 4]

# Bài 5
def count_changes_to_all_odd(nums1):
    if not any(x % 2 for x in nums1):
        return -1
    return sum(1 for x in nums1 if x % 2 == 0)
assert count_changes_to_all_odd([2, 3, 8]) == 2 and count_changes_to_all_odd([2, 4]) == -1
assert count_changes_to_all_odd([1, 3]) == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bài construct luôn đúng":**

```python
# 1. Thử vài ví dụ nhỏ bằng vét cạn để đoán đáp án
# 2. Chia trường hợp theo cấu trúc (có / không có số lẻ)
# 3. Chỉ ra phép dựng cụ thể cho từng trường hợp
# 4. Kết luận đáp án (ở đây): return True
```

**Checklist trước khi submit:**

- [ ] Đã nhớ `chẵn - lẻ` và `lẻ - chẵn` đều là **lẻ**, `lẻ - lẻ` là **chẵn**?
- [ ] Phép trừ dùng giá trị **gốc** `nums1[j]` với `j != i`?
- [ ] Đã xét `n = 1` (một phần tử luôn đồng nhất)?
- [ ] Đã kiểm chứng bằng vét cạn với `n` nhỏ trước khi tin vào kết luận `True`?
- [ ] Chữ ký đúng `uniformArray(self, nums1: List[int]) -> bool`?